In [1]:
# ==============================================================
# Cell 0: Mount Drive + imports + configuration + download helpers
# ==============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
import psutil
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
import joblib

from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_classif, f_classif, SelectKBest
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore')

# ==============================================================
# Path configuration
# ==============================================================

# Main output location: final cleaned dataset and cache
OUTPUT_PATH = "/content/drive/My Drive/NIDS_Project/output"
TEMP_PATH   = "/content/drive/My Drive/NIDS_Project/temp_preprocessed"

# Local Colab storage: used only for temporary raw file downloads
LOCAL_TEMP_RAW    = "/content/temp_raw_downloads"

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TEMP_PATH, exist_ok=True)
os.makedirs(LOCAL_TEMP_RAW, exist_ok=True)

# ==============================================================
# Pipeline configuration
# ==============================================================
RANDOM_STATE        = 42
CHUNK_SIZE          = 100000
SAMPLE_PER_LABEL    = 2000
MAX_SAMPLE_TOTAL    = 500000
CORR_THRESHOLD      = 0.95
VARIANCE_THRESHOLD  = 1e-6
MERGE_ATTEMPTED_TO_BENIGN = True  # Attempted-relabel-as-Benign -> Benign

# ==============================================================
# Shared download links for the raw CSE-CIC-IDS2018 parquet files
# ==============================================================
SHARED_LINKS = {
    "Botnet-Friday-02-03-2018.parquet":        "https://drive.google.com/file/d/13I3OdRW84KQZCgxJ0w90U9s1wkbifNvb/view?usp=drive_link",
    "Bruteforce-Wednesday-14-02-2018.parquet": "https://drive.google.com/file/d/1TN98X7LMB-k59AwCxh1WRVUsqWomculy/view?usp=drive_link",
    "DDoS1-Tuesday-20-02-2018.parquet":        "https://drive.google.com/file/d/1tAIUBeoK3XmBMgbW1DSITOKa2V-hJ42g/view?usp=drive_link",
    "DDoS2-Wednesday-21-02-2018.parquet":      "https://drive.google.com/file/d/18a4vy43hdiIWb4SkYT2Yoxf4an4cvjnc/view?usp=drive_link",
    "DoS2-Friday-16-02-2018.parquet":          "https://drive.google.com/file/d/1WqrW_vzFkC84feYOq0zUiGHr5bhETuZZ/view?usp=drive_link",
    "Dos1-Thursday-15-02-2018.parquet":        "https://drive.google.com/file/d/1ludaqAhCt9QjxqlUfYbhyMFksWEBmmB3/view?usp=drive_link",
    "Infil1-Wednesday-28-02-2018.parquet":     "https://drive.google.com/file/d/1N5np_Cih4YQyO_KSwCpFL-BKFfNUXVUa/view?usp=drive_link",
    "Infil2-Thursday-01-03-2018.parquet":      "https://drive.google.com/file/d/1zRY6yyV6kTQ2BQ3WkgpkgUhQlm9BIehd/view?usp=drive_link",
    "Web1-Thursday-22-02-2018.parquet":        "https://drive.google.com/file/d/16Y8e5xaaTSYqstwm0i42xAsQxrKL2_g5/view?usp=drive_link",
    "Web2-Friday-23-02-2018.parquet":          "https://drive.google.com/file/d/1TCYkzrjtx8HRQmy_B-jYxjDnUSm4MWv0/view?usp=drive_link",
}

print(f"Registered files for download: {len(SHARED_LINKS)}")
print(f"Final output directory (main Drive): {OUTPUT_PATH}")
print(f"Temporary cache directory (main Drive): {TEMP_PATH}")
print(f"Temporary download directory (Colab local): {LOCAL_TEMP_RAW}")
print(f"Settings: Sample/Label={SAMPLE_PER_LABEL}, MaxSample={MAX_SAMPLE_TOTAL}, Corr>{CORR_THRESHOLD}")


# --- Extract the Google Drive file ID from a shared link ---
def extract_file_id(shared_url):
    """Extract the Google Drive file ID from a shared link in any of its common formats."""
    patterns = [
        r'/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'open\?id=([a-zA-Z0-9_-]+)',
    ]
    for pat in patterns:
        match = re.search(pat, shared_url)
        if match:
            return match.group(1)
    return None


# --- Download a file from Google Drive ---
def download_gdrive_file(shared_url, output_path, filename):
    """Download a file from a shared Google Drive link to output_path."""
    file_id = extract_file_id(shared_url)
    if not file_id:
        raise ValueError(f"Could not extract a file ID from: {shared_url}")

    direct_url = f"https://drive.google.com/uc?id={file_id}&export=download"

    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024**2)
        print(f"   Already exists ({size_mb:.1f} MB): {filename}")
        return True

    print(f"   Downloading: {filename}")

    try:
        import gdown
        gdown.download(direct_url, output_path, quiet=False, fuzzy=True)
        if os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
            size_mb = os.path.getsize(output_path) / (1024**2)
            print(f"   Download complete: {size_mb:.1f} MB")
            return True
    except Exception as e:
        print(f"   gdown failed ({e}), retrying with requests...")

    try:
        session = requests.Session()
        response = session.get(direct_url, stream=True)
        for key in response.cookies:
            if key.name.startswith('download_warning'):
                direct_url = f"https://drive.google.com/uc?export=download&confirm={key.value}&id={file_id}"
                response = session.get(direct_url, stream=True)
                break

        total_size = int(response.headers.get('content-length', 0))
        with open(output_path, 'wb') as f:
            downloaded = 0
            for chunk in response.iter_content(8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
        size_mb = os.path.getsize(output_path) / (1024**2)
        print(f"   Download complete: {size_mb:.1f} MB")
        return True
    except Exception as e:
        print(f"   Error: {e}")
        if os.path.exists(output_path):
            os.remove(output_path)
        return False


# --- Log current RAM usage ---
def log_ram(step_name=""):
    ram = psutil.virtual_memory()
    used_gb = ram.used / (1024**3)
    total_gb = ram.total / (1024**3)
    print(f"   RAM [{step_name}]: {ram.percent}% ({used_gb:.1f}GB / {total_gb:.1f}GB)")


print("\nCell 0 complete. Ready to download and process.")
print("Reminder: replace the entries in SHARED_LINKS with your own file links before running.")


Mounted at /content/drive
Registered files for download: 10
Final output directory (main Drive): /content/drive/My Drive/NIDS_Project/output
Temporary cache directory (main Drive): /content/drive/My Drive/NIDS_Project/temp_preprocessed
Temporary download directory (Colab local): /content/temp_raw_downloads
Settings: Sample/Label=2000, MaxSample=500000, Corr>0.95

Cell 0 complete. Ready to download and process.
Reminder: replace the entries in SHARED_LINKS with your own file links before running.


In [2]:
# ==============================================================
# Cell 1: Download -> clean -> save to main Drive -> remove temp file
# ==============================================================

def clean_and_save(shared_name, shared_url):
    """
    1) Download the raw file from its shared link to local Colab storage
    2) Run the full cleaning pipeline
    3) Save the cleaned file to the main Google Drive output
    4) Delete the raw local file and free RAM
    """
    local_raw_path   = os.path.join(LOCAL_TEMP_RAW, shared_name)
    drive_clean_path = os.path.join(TEMP_PATH, shared_name.replace('.parquet', '_cleaned.parquet'))

    # Skip if the cleaned file already exists on Drive
    if os.path.exists(drive_clean_path):
        size_mb = os.path.getsize(drive_clean_path) / (1024**2)
        print(f"\n[SKIP] Already exists: {shared_name} ({size_mb:.1f} MB)")
        return {
            'file': shared_name,
            'status': 'skipped',
            'cleaned_path': drive_clean_path
        }

    print(f"\n{'='*60}")
    print(f"Starting: {shared_name}")
    log_ram("before download")

    # --- 1) Download ---
    success = download_gdrive_file(shared_url, local_raw_path, shared_name)
    if not success:
        return {'file': shared_name, 'status': 'download_failed'}
    log_ram("after download")

    # --- 2) Clean ---
    print(f"   Cleaning...")
    try:
        df = pd.read_parquet(local_raw_path)
        original_rows = len(df)
        print(f"      Initial rows: {original_rows:,}")

        # 2a) Infinity -> NaN
        df = df.replace([np.inf, -np.inf], np.nan)

        # 2b) Drop all-NaN rows
        df = df.dropna(how='all')

        # 2c) Drop duplicates
        df = df.drop_duplicates()
        after_dup = len(df)
        print(f"      Duplicates removed: {original_rows - after_dup:,} rows")

        # 2d) Merge the "Attempted-relabel-as-Benign" class into Benign
        if MERGE_ATTEMPTED_TO_BENIGN and 'Label' in df.columns:
            n_attempted = (df['Label'] == 'Attempted-relabel-as-Benign').sum()
            if n_attempted > 0:
                df['Label'] = df['Label'].replace('Attempted-relabel-as-Benign', 'Benign')
                print(f"      {n_attempted:,} 'Attempted...' rows relabeled as 'Benign'")

        # 2e) Drop identifier / non-numeric / text columns
        cols_to_drop = [
            'Flow ID', 'Src IP', 'Dst IP', 'Source IP', 'Destination IP',
            'Src Port', 'Dst Port', 'Source Port', 'Destination Port',
            'Protocol', 'Timestamp', 'SimillarHTTP'
        ]
        existing_drop = [c for c in cols_to_drop if c in df.columns]
        if existing_drop:
            df = df.drop(columns=existing_drop)
            print(f"      Identifier columns dropped: {existing_drop}")

        # Drop any remaining object columns (other than Label)
        obj_cols = df.select_dtypes(include=['object']).columns.tolist()
        if 'Label' in obj_cols:
            obj_cols.remove('Label')
        if obj_cols:
            df = df.drop(columns=obj_cols)
            print(f"      Extra object columns dropped: {obj_cols}")

        # 2f) Downcast to float32 (roughly 50% RAM savings)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], downcast='float')

        # --- 3) Save to main Drive ---
        df.to_parquet(drive_clean_path, index=False, compression='snappy')
        final_rows = len(df)
        final_cols = len(df.columns)
        print(f"      Saved to main Drive: {final_rows:,} x {final_cols}")
        log_ram("after cleaning")

        # Free the DataFrame
        del df
        gc.collect()

        # --- 4) Remove the raw file from local Colab storage ---
        if os.path.exists(local_raw_path):
            os.remove(local_raw_path)
            print(f"      Raw file removed from Colab local storage")
        log_ram("after removing raw file")

        return {
            'file': shared_name,
            'status': 'success',
            'original_rows': original_rows,
            'final_rows': final_rows,
            'final_cols': final_cols,
            'cleaned_path': drive_clean_path
        }

    except Exception as e:
        print(f"      Error: {e}")
        return {'file': shared_name, 'status': f'error: {str(e)}'}


# ==============================================================
# Run the pipeline over all registered files
# ==============================================================
print("Starting pipeline: download -> clean -> save to main Drive")
print("="*60)

results = []
for filename, link in tqdm(SHARED_LINKS.items(), desc="Processing files"):
    result = clean_and_save(filename, link)
    results.append(result)

    if result['status'] == 'success':
        print(f"   Success: {result['final_rows']:,} rows")
    elif result['status'] == 'skipped':
        print(f"   Skipped")
    else:
        print(f"   Failed: {result['status']}")

# Save summary statistics
with open(os.path.join(OUTPUT_PATH, 'stage1_cleaning_stats.json'), 'w') as f:
    json.dump(results, f, indent=2)

# Final report
success = sum(1 for r in results if r['status'] == 'success')
skipped = sum(1 for r in results if r['status'] == 'skipped')
failed  = len(results) - success - skipped

total_orig = sum(r.get('original_rows', 0) for r in results if r['status']=='success')
total_fin  = sum(r.get('final_rows', 0) for r in results if r['status']=='success')

print("\n" + "="*60)
print("Stage 1 complete.")
print(f"   Success: {success} | Already present: {skipped} | Failed: {failed}")
print(f"   Original rows: {total_orig:,}")
print(f"   Cleaned rows:  {total_fin:,}")
print(f"   Output directory: {OUTPUT_PATH}")
print("="*60)

print("\nFiles saved to main Drive:")
for f in sorted(os.listdir(OUTPUT_PATH)):
    if f.endswith('_cleaned.parquet'):
        size_mb = os.path.getsize(os.path.join(OUTPUT_PATH, f)) / (1024**2)
        print(f"   - {f} ({size_mb:.1f} MB)")


Starting pipeline: download -> clean -> save to main Drive


Processing files:   0%|          | 0/10 [00:00<?, ?it/s]


Starting: Botnet-Friday-02-03-2018.parquet
   RAM [before download]: 10.8% (1.1GB / 12.7GB)
   Downloading: Botnet-Friday-02-03-2018.parquet


Downloading...
From (original): https://drive.google.com/uc?id=13I3OdRW84KQZCgxJ0w90U9s1wkbifNvb
From (redirected): https://drive.google.com/uc?id=13I3OdRW84KQZCgxJ0w90U9s1wkbifNvb&confirm=t&uuid=b8c16383-0eb1-417c-a4a3-45fea19f9a8f
To: /content/temp_raw_downloads/Botnet-Friday-02-03-2018.parquet

  0%|          | 0.00/715M [00:00<?, ?B/s]
  1%|          | 4.72M/715M [00:00<00:47, 15.0MB/s]
  3%|▎         | 22.5M/715M [00:00<00:10, 65.6MB/s]
  5%|▍         | 33.6M/715M [00:00<00:13, 50.7MB/s]
  6%|▌         | 44.6M/715M [00:00<00:10, 63.7MB/s]
  8%|▊         | 54.0M/715M [00:01<00:13, 50.7MB/s]
 10%|█         | 71.8M/715M [00:01<00:11, 54.9MB/s]
 12%|█▏        | 82.3M/715M [00:01<00:10, 63.2MB/s]
 13%|█▎        | 94.9M/715M [00:01<00:10, 61.1MB/s]
 16%|█▌        | 111M/715M [00:01<00:07, 79.4MB/s] 
 17%|█▋        | 121M/715M [00:01<00:08, 72.2MB/s]
 19%|█▊        | 133M/715M [00:02<00:07, 80.7MB/s]
 22%|██▏       | 154M/715M [00:02<00:05, 111MB/s] 
 24%|██▍       | 170M/715M [00:02<00:

   Download complete: 682.0 MB
   RAM [after download]: 11.3% (1.1GB / 12.7GB)
   Cleaning...
      Initial rows: 5,094,963
      Duplicates removed: 0 rows
      208 'Attempted...' rows relabeled as 'Benign'
      Identifier columns dropped: ['Protocol']


...

Downloading...
From (original): https://drive.google.com/uc?id=1TCYkzrjtx8HRQmy_B-jYxjDnUSm4MWv0
From (redirected): https://drive.google.com/uc?id=1TCYkzrjtx8HRQmy_B-jYxjDnUSm4MWv0&confirm=t&uuid=358a0d57-a70a-4bbf-a292-76d3535efcc9
To: /content/temp_raw_downloads/Web2-Friday-23-02-2018.parquet

  0%|          | 0.00/655M [00:00<?, ?B/s]
  1%|          | 6.82M/655M [00:00<00:15, 43.1MB/s]
  4%|▍         | 25.7M/655M [00:00<00:05, 111MB/s] 
  6%|▌         | 38.8M/655M [00:00<00:07, 85.6MB/s]
  8%|▊         | 55.1M/655M [00:00<00:05, 104MB/s] 
 11%|█▏        | 73.9M/655M [00:00<00:04, 125MB/s]
 14%|█▍        | 92.8M/655M [00:00<00:04, 138MB/s]
 17%|█▋        | 114M/655M [00:00<00:03, 153MB/s] 
 20%|█▉        | 130M/655M [00:01<00:03, 139MB/s]
 22%|██▏       | 145M/655M [00:01<00:03, 138MB/s]
 25%|██▌       | 164M/655M [00:01<00:03, 145MB/s]
 27%|██▋       | 179M/655M [00:01<00:03, 146MB/s]
 30%|██▉       | 195M/655M [00:01<00:03, 134MB/s]
 32%|███▏      | 209M/655M [00:01<00:03, 119MB/s]

   Download complete: 624.7 MB
   RAM [after download]: 16.1% (1.7GB / 12.7GB)
   Cleaning...
      Initial rows: 4,565,131
      Duplicates removed: 0 rows
      72 'Attempted...' rows relabeled as 'Benign'
      Identifier columns dropped: ['Protocol']
      Saved to main Drive: 4,565,131 x 82
   RAM [after cleaning]: 26.2% (3.0GB / 12.7GB)


Processing files: 100%|██████████| 10/10 [18:09<00:00, 108.93s/it]

      Raw file removed from Colab local storage
   RAM [after removing raw file]: 16.3% (1.8GB / 12.7GB)
   Success: 4,565,131 rows



Stage 1 complete.
   Success: 10 | Already present: 0 | Failed: 0
   Original rows: 48,991,415
   Cleaned rows:  48,991,415
   Output directory: /content/drive/My Drive/NIDS_Project/output

Files saved to main Drive:
   - CICIDS2018_sampled_cleaned.parquet (1091.6 MB)


In [3]:
# ==============================================================
# Cell 2: Stratified sampling
# ==============================================================

def get_stratified_sample(cleaned_files, sample_per_label=2000, max_total=500000):
    """
    Draw up to sample_per_label rows per label from each cleaned file.
    This ensures that even rare labels (e.g. Web Attack-SQL) are represented
    in the sample used for feature selection.
    """
    all_samples = []

    for f in tqdm(cleaned_files, desc="Sampling"):
        path = os.path.join(TEMP_PATH, f)
        df = pd.read_parquet(path, columns=['Label'])
        labels = df['Label'].unique()
        del df

        for lbl in labels:
            # Read only the rows for this label (via filter pushdown)
            df_lbl = pd.read_parquet(path, filters=[('Label', '==', lbl)])
            n = min(sample_per_label, len(df_lbl))
            if n > 0:
                samp = df_lbl.sample(n=n, random_state=RANDOM_STATE)
                all_samples.append(samp)
            del df_lbl
            gc.collect()

    sample_df = pd.concat(all_samples, ignore_index=True)
    sample_df = sample_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    if len(sample_df) > max_total:
        sample_df = sample_df.sample(n=max_total, random_state=RANDOM_STATE)

    return sample_df


cleaned_files = sorted([f for f in os.listdir(TEMP_PATH) if f.endswith('_cleaned.parquet')])
print("Building a stratified sample for feature selection...")

sample_df = get_stratified_sample(cleaned_files, SAMPLE_PER_LABEL, MAX_SAMPLE_TOTAL)

print(f"\nSample ready: {len(sample_df):,} rows")
print(f"Label distribution in the sample:")
print(sample_df['Label'].value_counts())
print(f"\nSample size in RAM: {sample_df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# Save the sample (optional)
sample_df.to_parquet(os.path.join(TEMP_PATH, 'stratified_sample.parquet'), index=False)


Building a stratified sample for feature selection...


Sampling: 100%|██████████| 10/10 [02:45<00:00, 16.60s/it]



Sample ready: 41,098 rows
Label distribution in the sample:
Label
Benign                                          20000
Infiltration - NMAP Portscan                     4000
DDoS-LOIC-UDP                                    2527
SSH-BruteForce                                   2000
DDoS-LOIC-HTTP                                   2000
DoS GoldenEye                                    2000
Botnet Ares                                      2000
DoS Hulk                                         2000
DoS Slowloris                                    2000
DDoS-HOIC                                        2000
Infiltration - Communication Victim Attacker      203
Web Attack - Brute Force                          131
Web Attack - XSS                                  113
Infiltration - Dropbox Download                    85
Web Attack - SQL                                   39
Name: count, dtype: int64

Sample size in RAM: 0.01 GB


In [4]:
# ==============================================================
# Cell 3: Feature selection (hybrid: variance + correlation + tree + MI + ANOVA)
# ==============================================================

# --- Split into X and y ---
y_sample = sample_df['Label'].copy()
X_sample = sample_df.drop(columns=['Label']).copy()

# Guard against any remaining NaNs
X_sample = X_sample.fillna(0)

feature_names = X_sample.columns.tolist()
print(f"Initial feature count: {len(feature_names)}")

# --- 1) Variance threshold ---
print("\n[1/4] Variance threshold...")
selector_var = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
selector_var.fit(X_sample)
var_mask = selector_var.get_support()
features_after_var = [f for f, m in zip(feature_names, var_mask) if m]
print(f"   Features after variance filtering: {len(features_after_var)}")

X_var = X_sample[features_after_var].copy()

# --- 2) Correlation analysis (> 0.95) ---
print(f"\n[2/4] Correlation analysis (threshold={CORR_THRESHOLD})...")
corr_matrix = X_var.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
features_after_corr = [f for f in features_after_var if f not in to_drop_corr]
print(f"   Correlated features dropped: {len(to_drop_corr)}")
print(f"   Features after correlation filtering: {len(features_after_corr)}")

X_corr = X_var[features_after_corr].copy()
del X_var, corr_matrix, upper
gc.collect()

# --- 3) Hybrid feature selection ---
print(f"\n[3/4] Hybrid feature selection (tree + mutual information + ANOVA)...")

# Encode y numerically for the scoring methods below
le = LabelEncoder()
y_encoded = le.fit_transform(y_sample)

# 3a) Tree-based importance (Random Forest)
print("   Random Forest importance...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,        # bounded depth for speed
    n_jobs=-1,
    random_state=RANDOM_STATE,
    max_samples=0.3      # 30% of samples per tree for faster training
)
rf.fit(X_corr, y_encoded)
importances = rf.feature_importances_
tree_selected = [f for f, imp in zip(X_corr.columns, importances) if imp > np.percentile(importances, 50)]
print(f"      Tree-based selection: {len(tree_selected)} features")

# 3b) Mutual information
print("   Mutual information...")
mi_scores = mutual_info_classif(X_corr, y_encoded, random_state=RANDOM_STATE, n_jobs=-1)
mi_selected = [f for f, score in zip(X_corr.columns, mi_scores) if score > np.percentile(mi_scores, 50)]
print(f"      Mutual information selection: {len(mi_selected)} features")

# 3c) ANOVA F-test
print("   ANOVA F-test...")
f_scores, _ = f_classif(X_corr, y_encoded)
f_selected = [f for f, score in zip(X_corr.columns, f_scores) if score > np.percentile(f_scores, 50)]
print(f"      ANOVA selection: {len(f_selected)} features")

# 3d) Union of all three methods
final_features = list(set(tree_selected + mi_selected + f_selected))
final_features = sorted(final_features)  # sorted for reproducibility

print(f"\n   Final selected features: {len(final_features)}")
print(f"   Dimensionality reduction: {len(feature_names)} -> {len(final_features)} ({len(final_features)/len(feature_names)*100:.1f}%)")

# Show the top 20 features by Random Forest importance
print("\nTop 20 features by Random Forest importance:")
top_indices = np.argsort(importances)[::-1][:20]
for idx in top_indices:
    print(f"   - {X_corr.columns[idx]}: {importances[idx]:.4f}")

# Save the feature lists and label encoder
selected_data = {
    'final_features': final_features,
    'all_original_features': feature_names,
    'dropped_by_variance': [f for f in feature_names if f not in features_after_var],
    'dropped_by_correlation': to_drop_corr,
    'tree_selected': tree_selected,
    'mi_selected': mi_selected,
    'f_selected': f_selected,
    'label_classes': list(le.classes_)
}

with open(os.path.join(OUTPUT_PATH, 'selected_features.json'), 'w') as f:
    json.dump(selected_data, f, indent=2)

joblib.dump(le, os.path.join(OUTPUT_PATH, 'label_encoder.pkl'))

# Free memory
del X_corr, X_sample, y_sample, sample_df, rf
gc.collect()

print("\nFeature selection complete.")
print(f"Results saved to: {OUTPUT_PATH}")


Initial feature count: 81

[1/4] Variance threshold...
   Features after variance filtering: 80

[2/4] Correlation analysis (threshold=0.95)...
   Correlated features dropped: 28
   Features after correlation filtering: 52

[3/4] Hybrid feature selection (tree + mutual information + ANOVA)...
   Random Forest importance...
      Tree-based selection: 26 features
   Mutual information...
      Mutual information selection: 26 features
   ANOVA F-test...
      ANOVA selection: 26 features

   Final selected features: 38
   Dimensionality reduction: 81 -> 38 (46.9%)

Top 20 features by Random Forest importance:
   - Init Fwd Win Bytes: 0.0810
   - Fwd Seg Size Min: 0.0738
   - Bwd Packet Length Max: 0.0562
   - Bwd Packet Length Mean: 0.0483
   - Fwd Packet Length Max: 0.0472
   - Packet Length Variance: 0.0416
   - Fwd Packet Length Mean: 0.0402
   - Total Fwd Packets: 0.0392
   - Total Backward Packets: 0.0380
   - Bwd PSH Flags: 0.0332
   - Packet Length Std: 0.0306
   - Packet Length 

In [5]:
# ==============================================================
# Cell 4: Build the final merged dataset
# ==============================================================

# Load the selected feature list
with open(os.path.join(OUTPUT_PATH, 'selected_features.json'), 'r') as f:
    selected_data = json.load(f)

FINAL_FEATURES = selected_data['final_features']
KEEP_COLS = FINAL_FEATURES + ['Label']

print(f"Final columns: {len(KEEP_COLS)} ({len(FINAL_FEATURES)} features + Label)")

# --- Fit the scaler on the sample ---
print("\nFitting StandardScaler on the sample...")
sample_for_scaler = pd.read_parquet(os.path.join(TEMP_PATH, 'stratified_sample.parquet'))
sample_for_scaler = sample_for_scaler[KEEP_COLS].fillna(0)

X_samp = sample_for_scaler.drop('Label', axis=1).values
scaler = StandardScaler()
scaler.fit(X_samp)

joblib.dump(scaler, os.path.join(OUTPUT_PATH, 'standard_scaler.pkl'))
print("   Scaler saved.")

del sample_for_scaler, X_samp
gc.collect()

# --- Write the final dataset incrementally ---
final_parquet_path = os.path.join(OUTPUT_PATH, 'CICIDS2018_processed_final.parquet')
final_csv_path = os.path.join(OUTPUT_PATH, 'CICIDS2018_processed_final.csv')

print(f"\nBuilding the final Parquet file...")
print(f"   Path: {final_parquet_path}")

writer = None
total_rows = 0
label_dist = {}

cleaned_files = sorted([f for f in os.listdir(TEMP_PATH) if f.endswith('_cleaned.parquet')])

for f in tqdm(cleaned_files, desc="Processing files"):
    path = os.path.join(TEMP_PATH, f)

    # Read the cleaned file
    df = pd.read_parquet(path)

    # Keep only the selected features
    missing_cols = [c for c in KEEP_COLS if c not in df.columns]
    if missing_cols:
        print(f"   Missing columns in {f}: {missing_cols}")
        for c in missing_cols:
            df[c] = 0

    df = df[KEEP_COLS].copy()

    # Fill remaining NaNs
    df[FINAL_FEATURES] = df[FINAL_FEATURES].fillna(0)

    # Normalize
    X_num = df[FINAL_FEATURES].values
    X_scaled = scaler.transform(X_num)
    df[FINAL_FEATURES] = X_scaled.astype(np.float32)

    # Track label distribution
    vc = df['Label'].value_counts()
    for lbl, cnt in vc.items():
        label_dist[lbl] = label_dist.get(lbl, 0) + cnt

    total_rows += len(df)

    # Incremental write with PyArrow
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(final_parquet_path, table.schema, compression='snappy')
    writer.write_table(table)

    # Free memory
    del df, X_num, X_scaled, table
    gc.collect()

if writer:
    writer.close()

print(f"\n{'='*60}")
print(f"Final file built.")
print(f"   Total rows: {total_rows:,}")
print(f"   Path: {final_parquet_path}")
print(f"   File size: {os.path.getsize(final_parquet_path) / 1024**3:.2f} GB")

print(f"\nFinal label distribution:")
for lbl, cnt in sorted(label_dist.items(), key=lambda x: -x[1]):
    pct = cnt / total_rows * 100
    print(f"   - {lbl}: {cnt:,} ({pct:.2f}%)")

# --- Optional CSV export ---
# Recommendation: only export a small sample to CSV. Parquet is sufficient
# for the rest of the pipeline. To export the full dataset to CSV instead:
# df_final = pd.read_parquet(final_parquet_path)
# df_final.to_csv(final_csv_path, index=False)
# print(f"   CSV also saved: {final_csv_path}")

print("="*60)


Final columns: 39 (38 features + Label)

Fitting StandardScaler on the sample...
   Scaler saved.

Building the final Parquet file...
   Path: /content/drive/My Drive/NIDS_Project/output/CICIDS2018_processed_final.parquet


Processing files: 100%|██████████| 10/10 [05:45<00:00, 34.60s/it]


Final file built.
   Total rows: 48,991,415
   Path: /content/drive/My Drive/NIDS_Project/output/CICIDS2018_processed_final.parquet
   File size: 3.55 GB

Final label distribution:
   - Benign: 45,530,463 (92.94%)
   - DoS Hulk: 1,803,160 (3.68%)
   - DDoS-HOIC: 1,082,291 (2.21%)
   - DDoS-LOIC-HTTP: 289,302 (0.59%)
   - Botnet Ares: 142,921 (0.29%)
   - SSH-BruteForce: 94,197 (0.19%)
   - DoS GoldenEye: 22,559 (0.05%)
   - Infiltration - NMAP Portscan: 14,934 (0.03%)
   - DoS Slowloris: 8,490 (0.02%)
   - DDoS-LOIC-UDP: 2,527 (0.01%)
   - Infiltration - Communication Victim Attacker: 203 (0.00%)
   - Web Attack - Brute Force: 131 (0.00%)
   - Web Attack - XSS: 113 (0.00%)
   - Infiltration - Dropbox Download: 85 (0.00%)
   - Web Attack - SQL: 39 (0.00%)


In [6]:
# ==============================================================
# Cell 5: Final report and cleanup
# ==============================================================

import os
import json
import gc
import pandas as pd

print("CICIDS2018 pipeline report")
print("="*60)

# 1) Feature selection summary (if the file exists)
selected_json_path = os.path.join(OUTPUT_PATH, 'selected_features.json')
if os.path.exists(selected_json_path):
    with open(selected_json_path, 'r') as f:
        selected_data = json.load(f)

    print(f"\nFeature selection:")
    if 'all_original_features' in selected_data:
        print(f"   Original features: {len(selected_data['all_original_features'])}")
    if 'final_features' in selected_data:
        print(f"   Final features: {len(selected_data['final_features'])}")
    if 'dropped_by_variance' in selected_data:
        print(f"   Dropped by variance filter: {len(selected_data['dropped_by_variance'])}")
    if 'dropped_by_correlation' in selected_data:
        print(f"   Dropped by correlation filter: {len(selected_data['dropped_by_correlation'])}")
    if 'tree_selected' in selected_data:
        print(f"   Selected by tree importance: {len(selected_data['tree_selected'])}")
    if 'mi_selected' in selected_data:
        print(f"   Selected by mutual information: {len(selected_data['mi_selected'])}")
    if 'f_selected' in selected_data:
        print(f"   Selected by ANOVA: {len(selected_data['f_selected'])}")

# 2) List output files
print(f"\nOutput files in: {OUTPUT_PATH}")
if os.path.exists(OUTPUT_PATH):
    for f in sorted(os.listdir(OUTPUT_PATH)):
        size_mb = os.path.getsize(os.path.join(OUTPUT_PATH, f)) / (1024**2)
        print(f"   - {f} ({size_mb:.1f} MB)")

# 3) Sanity check on the final file
if 'final_parquet_path' in locals() and os.path.exists(final_parquet_path):
    df_check = pd.read_parquet(final_parquet_path, columns=['Label'])
    print(f"\nFinal file sanity check:")
    print(f"   Row count: {len(df_check):,}")
    print(f"   Unique labels: {df_check['Label'].nunique()}")
    print(f"   Sample labels: {list(df_check['Label'].unique()[:10])}")
    del df_check
    gc.collect()

# --- Optional cleanup of temporary files ---
# if os.path.exists(TEMP_PATH):
#     import shutil
#     shutil.rmtree(TEMP_PATH)
#     print(f"\nTemporary files in {TEMP_PATH} removed.")

print("\n" + "="*60)
print("All stages completed successfully.")
print("="*60)


CICIDS2018 pipeline report

Feature selection:
   Original features: 81
   Final features: 38
   Dropped by variance filter: 1
   Dropped by correlation filter: 28
   Selected by tree importance: 26
   Selected by mutual information: 26
   Selected by ANOVA: 26

Output files in: /content/drive/My Drive/NIDS_Project/output
   - CICIDS2018_processed_final.parquet (3633.2 MB)
   - CICIDS2018_sampled_cleaned.parquet (1091.6 MB)
   - LightGBM_Classification_Report.txt (0.0 MB)
   - LightGBM_Confusion_Matrix.png (0.4 MB)
   - LightGBM_Feature_Importance.png (0.2 MB)
   - NIDS_Dimensionality_Reduction.png (2.4 MB)
   - NIDS_PCA_Single.png (0.3 MB)
   - NIDS_UMAP_Single.png (0.9 MB)
   - NIDS_t-SNE_Single.png (1.2 MB)
   - label_encoder.pkl (0.0 MB)
   - label_mapping.json (0.0 MB)
   - lightgbm_nids_model.pkl (4.9 MB)
   - lightgbm_nids_model.txt (4.9 MB)
   - selected_features.json (0.0 MB)
   - stage1_cleaning_stats.json (0.0 MB)
   - standard_scaler.pkl (0.0 MB)

Final file sanity check:
 